# NEG-Net Tier-0 -- First Real Training Run (raw-DINOv3, Shard 1)

Runs the full NCR-Match pipeline (Stage 1 retrieval -> Stage 2 geometry filter -> Stage 3
VGGT/pose scoring -> graph assembly -> NEG-Net training) end-to-end using **raw DINOv3**
(`ModelComboDINOv3Raw`, no fusion head) as the canonical backbone for both retrieval and
NEG-Net's node features.

**This is a train-once run.** Per project decision, once this notebook produces frozen
checkpoints they are not retrained afterward -- they're meant to generalize to other
shards/datasets directly, not be iterated on. See the session plan
(`hi-yes-so-i-piped-nova.md`) for full background.

Structure:
1. Setup (deps, Drive mount, paths, helpers)
2. Load ground truth (fixed pairs for the smoke test)
3. **Smoke test** -- full pipeline, FRESH execution, tiny fixed subset. Must pass before
   the real run below it.
4. Stage 1 retrieval (reuse-or-run)
5. Stage 2 geometry filter + Stage 3 VGGT/pose scoring (reuse-or-run)
6. Graph assembly (build / join-labels / folds) + record the training-exposure registry
   (see `NCR/negnet_training_exposure/`)
7. Node features + `NODE_DIM` sanity check
8. Train NEG-Net: all 3 attribution-ladder configs, each frozen and saved separately
9. Report metrics side by side

**Prerequisites staged on Drive** (`DRIVE_ROOT = /content/drive/MyDrive/ImageSimilarity`):
- `scripts/` -- this project's `.py` files, INCLUDING the just-edited `train_negnet.py`
  (dynamic `NODE_DIM` inference instead of the old hardcoded 3024 -- re-upload it before
  running this notebook; cell 5 below verifies the live version actually has the fix)
- `NCR/ground_truth/ground_truth.csv` -- copy from the local repo's (gitignored) `NCR/`
  folder, same relative layout
- `NCR/shard_membership/source_shard_assignment.json` -- only needed if the Stage 2&3
  fallback branch below actually runs (i.e. Drive's reused rawDINO Shard 1 output is
  missing); not required otherwise
- `dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth`, `weights/outdoor.ckpt`,
  `weights/vggt_omega_1b_512.pt`, `scripts/ml-aspanformer/`
- `workingsourcecrops.zip`, `working_targetcrops.zip` (full image corpus)
- `ablation_outputs/retrieval/dinov3raw_retrieval_manifest.jsonl` +
  `dinov3raw_feature_cache.npz`, `ablation_outputs/vggt/dinov3raw_shard1/
  pose_scored_manifest.jsonl`, `ablation_outputs/geometry_filter/dinov3raw_shard1/
  vggt_candidates_manifest.jsonl` -- already staged from the earlier raw-DINOv3 ablation run
  (see `NCR/ablation_results/rawdino/`); if present, Stages 1/2+3 below just import them
  instead of re-running. The candidates file matters: `pose_scored_manifest.jsonl` alone is
  only the Stage-3 fresh-compute residual (98 rows), not the complete 731-row candidate set --
  Step 4 below combines both.


In [ ]:
# Step 0 -- Install dependencies and verify GPU
# einops pin: ml-aspanformer needs einops<0.6 (>=0.6 removed _torch_specific it depends on).
# opencv upgrade: Colab's opencv wheel is built against numpy 1.x; current PyTorch pins
# numpy 2.x. opencv-python-headless>=4.9.0 added numpy 2.x ABI support (vggt_signals.py
# needs this).
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "einops==0.3.0", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python-headless>=4.9.0", "-q"], check=True)

import torch, cv2, numpy as np
print(f"PyTorch : {torch.__version__}")
print(f"numpy   : {np.__version__}")
print(f"cv2     : {cv2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU -- switch runtime to GPU before continuing.")


In [ ]:
# Step 1 -- Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 2 -- Configure paths and shared helpers
# ── EDIT THESE ──────────────────────────────────────────────────────────────
from pathlib import Path
import json, shutil, sys, os, importlib, time

DRIVE_ROOT   = Path('/content/drive/MyDrive/ImageSimilarity')
SCRIPTS_DIR  = DRIVE_ROOT / 'scripts'
ASPANPATH    = '/content/ml-aspanformer'
WEIGHTS_PATH = DRIVE_ROOT / 'aspanweights' / 'outdoor.ckpt'
CONFIG_PATH  = '/content/ml-aspanformer/configs/aspan/outdoor/aspan_test.py'
VGGT_CHECKPOINT = DRIVE_ROOT / 'weights' / 'VGGT-Omega' / 'vggt_omega_1b_512.pt'
DINOV3_CHECKPOINT_DRIVE = DRIVE_ROOT / 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'

SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'
LOCAL_SOURCE_DIR = Path('./source')
LOCAL_TARGET_DIR = Path('./target')

# Stage 1 retrieval outputs (already produced by retrieval_ablation_colab.ipynb).
RETRIEVAL_OUT = DRIVE_ROOT / 'ablation_outputs' / 'retrieval'


def _find_existing(directory, *names):
    """First existing directory/name among names, in order, or None. Used where the
    exact extension actually staged on Drive isn't confirmed (e.g. the Stage 1 retrieval
    manifest was reported as .json, but every other manifest in this project is .jsonl) --
    checks explicitly instead of guessing one and silently falling through to a fresh run."""
    directory = Path(directory)
    for name in names:
        candidate = directory / name
        if candidate.exists():
            return candidate
    return None


DINOV3RAW_RETRIEVAL_MANIFEST = (
    _find_existing(RETRIEVAL_OUT, 'dinov3raw_retrieval_manifest.jsonl', 'dinov3raw_retrieval_manifest.json')
    or RETRIEVAL_OUT / 'dinov3raw_retrieval_manifest.jsonl'  # default if neither exists yet -- Step 1 falls back to a fresh run
)
DINOV3RAW_FEATURE_CACHE = RETRIEVAL_OUT / 'dinov3raw_feature_cache.npz'

# Stage 2/3 outputs (already produced by dinov3raw_jocch_ablation_colab.ipynb).
# pose_scored_manifest.jsonl's fields are a strict superset of vggt_judged_manifest.jsonl's
# (see pose_scoring.py's process_manifest: it copies every input field through and only ADDS
# true_match/judgement/reason/pose_scoring_config) -- but the FILE ITSELF is only the Stage-3
# fresh-compute residual (98 rows), not the complete candidate set: most of the real Shard 1
# candidates (631 more rows) inherited an already-known judgement from baseline's own review
# and never got a fresh Stage-3 row at all. vggt_candidates_manifest.jsonl below is the
# complete Filter-1-survivor set (731 rows, full ASpan-2D evidence on every row) that Step 4
# combines this file with -- see that cell for why both are needed.
DINOV3RAW_S1_POSE_MANIFEST = (DRIVE_ROOT / 'ablation_outputs' / 'vggt' / 'dinov3raw_shard1'
                               / 'pose_scored_manifest.jsonl')
DINOV3RAW_S1_VGGT_CANDIDATES = (DRIVE_ROOT / 'ablation_outputs' / 'geometry_filter' / 'dinov3raw_shard1'
                                 / 'vggt_candidates_manifest.jsonl')

# NCR/ artifacts -- local-only/gitignored in the repo, so they must be staged on Drive
# separately for Colab to see them. Upload with the same relative layout as the repo's
# (gitignored) NCR/ folder.
NCR_DRIVE = DRIVE_ROOT / 'NCR'
GROUND_TRUTH_CSV = NCR_DRIVE / 'ground_truth' / 'ground_truth.csv'
# Pure default/raw shard lookup (which physical shard a source belongs to) -- only consulted
# by the Stage 2&3 fallback branch below, unrelated to NEG-Net leakage prevention (that's
# NCR/negnet_training_exposure/, keyed on node exposure, no shard concept at all).
SOURCE_SHARD_ASSIGNMENT_JSON = NCR_DRIVE / 'shard_membership' / 'source_shard_assignment.json'

# This notebook's own outputs.
NEGNET_OUT = DRIVE_ROOT / 'ablation_outputs' / 'negnet_tier0_shard1'
NEGNET_CHECKPOINTS_DRIVE = NEGNET_OUT / 'checkpoints'
# ────────────────────────────────────────────────────────────────────────────

LOCAL_WORK = Path('/content/negnet_tier0_shard1')
(LOCAL_WORK / '_local').mkdir(parents=True, exist_ok=True)
(LOCAL_WORK / 'ablation').mkdir(parents=True, exist_ok=True)
NEGNET_OUT.mkdir(parents=True, exist_ok=True)


def sync_dir_to_drive(local_dir, drive_dir, filenames):
    '''Copy named files (if present) from local_dir to drive_dir, creating drive_dir
    if needed. Small manifests/checkpoints -- overwritten in full on every call.'''
    local_dir = Path(local_dir)
    drive_dir = Path(drive_dir)
    drive_dir.mkdir(parents=True, exist_ok=True)
    for name in filenames:
        src = local_dir / name
        if src.exists():
            shutil.copy2(src, drive_dir / name)
            print(f'  Saved: {name}')
        else:
            print(f'  MISSING (not produced): {name}')


def run_module_main(module, argv):
    '''Call module.main() with a temporary sys.argv override, for scripts
    (graph_assembly.py, train_negnet.py) whose main() reads sys.argv directly instead of
    accepting an argv parameter (unlike retrieve.py/geometry_filter.py/vggt_signals.py/
    pose_scoring.py, which all support main(argv=...)).'''
    old_argv = sys.argv
    sys.argv = ['notebook'] + list(argv)
    try:
        module.main()
    finally:
        sys.argv = old_argv


print(f"ASpanFormer path : {ASPANPATH}")
print(f"Weights          : {WEIGHTS_PATH}  (exists: {WEIGHTS_PATH.exists()})")
print(f"VGGT checkpoint  : {VGGT_CHECKPOINT}  (exists: {VGGT_CHECKPOINT.exists()})")
print(f"DINOv3 checkpoint: {DINOV3_CHECKPOINT_DRIVE}  (exists: {DINOV3_CHECKPOINT_DRIVE.exists()})")
print(f"Source zip       : {SOURCE_ZIP}  (exists: {SOURCE_ZIP.exists()})")
print(f"Target zip       : {TARGET_ZIP}  (exists: {TARGET_ZIP.exists()})")
print(f"Ground truth CSV : {GROUND_TRUTH_CSV}  (exists: {GROUND_TRUTH_CSV.exists()})")
print(f"Stage1 retrieval manifest resolved to: {DINOV3RAW_RETRIEVAL_MANIFEST}  (exists: {DINOV3RAW_RETRIEVAL_MANIFEST.exists()})")
print(f"Stage2+3 pose manifest (reuse candidate): {DINOV3RAW_S1_POSE_MANIFEST.exists()}")
print(f"Stage2+3 candidates manifest (reuse candidate, required alongside the pose manifest): {DINOV3RAW_S1_VGGT_CANDIDATES.exists()}")
print(f"Source shard assignment (fallback-branch only): {SOURCE_SHARD_ASSIGNMENT_JSON.exists()}")


In [ ]:
# Step 3 -- Copy pipeline scripts from Drive to Colab local filesystem
to_copy = [
    (SCRIPTS_DIR / 'retrieve.py',                        LOCAL_WORK / 'retrieve.py'),
    (SCRIPTS_DIR / '_local' / 'ModelComboDINOv3Raw.py',  LOCAL_WORK / '_local' / 'ModelComboDINOv3Raw.py'),
    (SCRIPTS_DIR / 'geometry_filter.py',                 LOCAL_WORK / 'geometry_filter.py'),
    (SCRIPTS_DIR / 'aspan_batching.py',                  LOCAL_WORK / 'aspan_batching.py'),
    (SCRIPTS_DIR / 'vggt_signals.py',                    LOCAL_WORK / 'vggt_signals.py'),
    (SCRIPTS_DIR / 'pose_scoring.py',                    LOCAL_WORK / 'pose_scoring.py'),
    (SCRIPTS_DIR / 'graph_assembly.py',                  LOCAL_WORK / 'graph_assembly.py'),
    (SCRIPTS_DIR / 'losses.py',                          LOCAL_WORK / 'losses.py'),
    (SCRIPTS_DIR / 'negnet.py',                          LOCAL_WORK / 'negnet.py'),
    (SCRIPTS_DIR / 'train_negnet.py',                    LOCAL_WORK / 'train_negnet.py'),
    (SCRIPTS_DIR / 'ablation_utils.py',     LOCAL_WORK / 'ablation' / 'ablation_utils.py'),
]

for src, dst in to_copy:
    if not src.exists():
        raise FileNotFoundError(f"Missing on Drive: {src}\nUpload it to {src.parent} and re-run.")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(src), str(dst))
    print(f"Copied: {src.name}")

# DINOv3-raw resolves its checkpoint next to itself first (see ModelComboDINOv3Raw.py's
# _resolve_checkpoint) -- staging it into _local/ alongside the script covers that.
shutil.copy(str(DINOV3_CHECKPOINT_DRIVE), str(LOCAL_WORK / '_local' / DINOV3_CHECKPOINT_DRIVE.name))
print(f"Copied checkpoint: {DINOV3_CHECKPOINT_DRIVE.name}")

sys.path.insert(0, str(LOCAL_WORK))
print("\nAll scripts ready.")


In [ ]:
# Step 3 (guard) -- verify the just-edited train_negnet.py (dynamic NODE_DIM inference)
# is the version actually staged on Drive. Checks the LIVE in-memory module's source, not
# just the file on disk -- catches "forgot to re-upload after editing" as well as "forgot
# to re-upload at all". This matters more than usual here: the old hardcoded NODE_DIM=3024
# would silently misshape raw DINOv3's 1024-d features instead of failing loudly, and this
# run's weights are meant to be frozen forever once trained.
import inspect
import train_negnet
importlib.reload(train_negnet)

_tn_src = inspect.getsource(train_negnet)
if 'NODE_DIM = 3024' in _tn_src or 'infer_node_dim' not in _tn_src:
    raise RuntimeError(
        "train_negnet.py on Drive is STILL THE OLD HARDCODED-NODE_DIM VERSION. "
        f"Loaded from: {train_negnet.__file__}\n"
        "Fix: re-upload the corrected train_negnet.py (with infer_node_dim(), no module-"
        "level NODE_DIM constant) to Drive scripts/, re-run the script-copy cell above, "
        "then re-run this cell."
    )
print("train_negnet.py: dynamic NODE_DIM inference verified present.")
print(f"  Loaded from: {train_negnet.__file__}")


In [ ]:
# Step 3a -- Copy ml-aspanformer from Drive; verify weights/checkpoints are staged
ASPAN_SRC = DRIVE_ROOT / 'ml-aspanformer'

if not Path(ASPANPATH).exists():
    if not ASPAN_SRC.exists():
        raise FileNotFoundError(
            f"ml-aspanformer not found on Drive at {ASPAN_SRC}\n"
            "Upload the ml-aspanformer/ directory there and re-run."
        )
    print(f"Copying ml-aspanformer from Drive -> {ASPANPATH} ...")
    shutil.copytree(str(ASPAN_SRC), ASPANPATH)
    print("Done.")
else:
    print(f"{ASPANPATH} already present.")

for label, path in (('ASpanFormer weights', WEIGHTS_PATH), ('VGGT checkpoint', VGGT_CHECKPOINT)):
    if not path.exists():
        raise FileNotFoundError(f"Missing on Drive: {path}")
    print(f"{label}: {path}")


In [ ]:
!pip install torchmetrics
!pip install albumentations
!pip install pytorch-lightning
!pip install -r ./ml-aspanformer/requirements.txt

In [ ]:
# Step 3b -- Install vggt_omega
VGGT_GIT_URL = "git+https://github.com/facebookresearch/vggt-omega.git"  # <- EDIT to the actual URL used previously

import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", VGGT_GIT_URL, "-q"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("vggt_omega install failed -- check VGGT_GIT_URL")
print("vggt_omega installed.")


In [ ]:
# Step 3c -- Extract and flatten image corpus from Drive zips (idempotent)
def _extract_and_flatten(zip_src, local_name):
    dst_dir = Path(f'./{local_name}')
    if dst_dir.exists() and any(dst_dir.iterdir()):
        n = sum(1 for p in dst_dir.iterdir() if p.is_file())
        print(f"  {local_name}/: already present ({n:,} files)")
        return n
    local_zip = Path(f'./{local_name}.zip')
    print(f"  Copying {Path(str(zip_src)).name} from Drive ...")
    shutil.copy(str(zip_src), str(local_zip))
    t0 = time.time()
    os.system(f'unzip -q {local_zip} -d {local_name}')
    print(f"  Unzipped in {time.time() - t0:.0f}s")
    for root, dirs, files in os.walk(str(dst_dir)):
        if root == str(dst_dir):
            continue
        for fname in files:
            src_path = os.path.join(root, fname)
            dst_path = os.path.join(str(dst_dir), fname)
            if os.path.exists(dst_path):
                base, ext = os.path.splitext(fname)
                dst_path = os.path.join(str(dst_dir), f"{base}_{root.replace('/', '_')}{ext}")
            shutil.move(src_path, dst_path)
    for root, dirs, files in os.walk(str(dst_dir), topdown=False):
        if root == str(dst_dir):
            continue
        try:
            os.rmdir(root)
        except OSError:
            pass
    local_zip.unlink(missing_ok=True)
    n = sum(1 for p in dst_dir.iterdir() if p.is_file())
    print(f"  {local_name}/: {n:,} files ready")
    return n

n_src = _extract_and_flatten(SOURCE_ZIP, 'source')
n_tgt = _extract_and_flatten(TARGET_ZIP, 'target')
print(f"\nImages ready: {n_src:,} sources, {n_tgt:,} targets")


In [ ]:
# Step 4 -- Load ground truth (source of the smoke test's fixed Positive pairs).
import csv


def load_ground_truth_positive_pairs(csv_path):
    pairs = {}
    with open(str(csv_path), newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if row['classification'] == 'Positive':
                pairs[(row['source_id'], row['target_id'])] = row
    return pairs


ground_truth_positive = load_ground_truth_positive_pairs(GROUND_TRUTH_CSV)
print(f"Ground truth: {len(ground_truth_positive)} Positive pairs")


## Smoke test -- run before the real Shard 1 pipeline below

Full pipeline, FRESH execution (no Drive reuse anywhere in this section), on a small fixed
subset that includes at least one confirmed Positive pair -- so label wiring through
`join-labels` is actually exercised, not just the retrieval/geometry/VGGT plumbing.
Purpose: catch shape mismatches (`NODE_DIM` inference), broken paths, and API drift cheaply,
before committing GPU time or touching the real, frozen-forever run.


In [ ]:
# Step 0.5a -- Build the smoke-test subset: 5 fixed Shard-1 Positive pairs from
# ground_truth.csv (deterministic: sorted, not random), so this cell picks the same images
# every run.
SMOKE_WORK = LOCAL_WORK / 'smoke'
SMOKE_SOURCE = SMOKE_WORK / 'source'
SMOKE_TARGET = SMOKE_WORK / 'target'
for d in (SMOKE_SOURCE, SMOKE_TARGET):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

shard1_positive_pairs = sorted(
    (sid, tid) for (sid, tid), row in ground_truth_positive.items()
    if row['shard'] == 'Shard1'
)[:5]
if not shard1_positive_pairs:
    raise RuntimeError("No Shard1 Positive pairs found in ground_truth.csv -- cannot build smoke test.")

print(f"Smoke test picks {len(shard1_positive_pairs)} Positive pairs:")
for sid, tid in shard1_positive_pairs:
    print(f"  {sid}  <->  {tid}")

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp', '.gif')


def _find_image(stem, root_dir):
    for ext in IMAGE_EXTS:
        candidate = root_dir / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    matches = list(root_dir.glob(f'{stem}.*'))
    return matches[0] if matches else None


n_found_source = n_found_target = 0
for sid, tid in shard1_positive_pairs:
    src_img = _find_image(sid, LOCAL_SOURCE_DIR)
    tgt_img = _find_image(tid, LOCAL_TARGET_DIR)
    if src_img:
        shutil.copy(str(src_img), str(SMOKE_SOURCE / src_img.name))
        n_found_source += 1
    else:
        print(f"  WARNING: source image not found for {sid}")
    if tgt_img:
        shutil.copy(str(tgt_img), str(SMOKE_TARGET / tgt_img.name))
        n_found_target += 1
    else:
        print(f"  WARNING: target image not found for {tid}")

print(f"\nSmoke subset: {n_found_source} source images, {n_found_target} target images copied.")
if n_found_source == 0 or n_found_target == 0:
    raise RuntimeError(
        "Could not find any smoke-test images in the extracted corpus -- check "
        "LOCAL_SOURCE_DIR/LOCAL_TARGET_DIR contents and ground_truth.csv id conventions."
    )


In [ ]:
# Step 0.5b -- Run the full pipeline FRESH on the smoke subset: retrieval -> geometry
# filter (lenient breakpoint, see below) -> VGGT -> pose scoring -> graph assembly ->
# abbreviated NEG-Net training. No Drive reuse anywhere in this cell.
SMOKE_TOPX = max(len(list(SMOKE_TARGET.glob('*'))), 1)
print(f"Smoke topx: {SMOKE_TOPX}")

import retrieve, geometry_filter, vggt_signals, pose_scoring, graph_assembly, train_negnet
for _m in (retrieve, geometry_filter, vggt_signals, pose_scoring, graph_assembly, train_negnet):
    importlib.reload(_m)

SMOKE_STAGE1_OUT = SMOKE_WORK / 'stage1'
retrieve.main([
    '--model-definition', str(LOCAL_WORK / '_local' / 'ModelComboDINOv3Raw.py'),
    '--weights',          'none',
    '--source',           str(SMOKE_SOURCE),
    '--target',           str(SMOKE_TARGET),
    '--output-dir',       str(SMOKE_STAGE1_OUT),
    '--topx',             str(SMOKE_TOPX),
    '--device',           'auto',
])
SMOKE_RETRIEVAL_MANIFEST = SMOKE_STAGE1_OUT / 'retrieval_manifest.jsonl'
SMOKE_FEATURE_CACHE = SMOKE_STAGE1_OUT / 'feature_cache.npz'

# Lenient breakpoint: the smoke test validates PLUMBING, not match quality -- the real
# breakpoint (50, paper default) could legitimately reject a pair on this tiny crop
# selection and mask a real bug behind "well, geometry just didn't pass." The real run
# below uses the paper default.
SMOKE_STAGE2_OUT = SMOKE_WORK / 'stage2'
geometry_filter.main([
    '--input-manifest',   str(SMOKE_RETRIEVAL_MANIFEST),
    '--output-dir',       str(SMOKE_STAGE2_OUT),
    '--breakpoint-value', '0',
    '--aspanpath',        ASPANPATH,
    '--weights_path',     str(WEIGHTS_PATH),
    '--config_path',      CONFIG_PATH,
    '--device',           'auto',
    '--progress-every',   '5',
])

SMOKE_STAGE3_VGGT_OUT = SMOKE_WORK / 'stage3_vggt'
vggt_signals.main([
    '--input-manifest', str(SMOKE_STAGE2_OUT / 'vggt_candidates_manifest.jsonl'),
    '--output-dir',     str(SMOKE_STAGE3_VGGT_OUT),
    '--checkpoint',     str(VGGT_CHECKPOINT),
    '--device',         'auto',
    '--progress-every', '1',
])

SMOKE_STAGE3_POSE_OUT = SMOKE_WORK / 'stage3_pose'
pose_scoring.main([
    '--input-manifest', str(SMOKE_STAGE3_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
    '--output-dir',     str(SMOKE_STAGE3_POSE_OUT),
])
SMOKE_JUDGE_MANIFEST = SMOKE_STAGE3_POSE_OUT / 'pose_scored_manifest.jsonl'

SMOKE_GRAPH_DIR = SMOKE_WORK / 'graph'
SMOKE_RAW_GRAPH = SMOKE_GRAPH_DIR / 'graph.jsonl'
SMOKE_LABELED_GRAPH = SMOKE_GRAPH_DIR / 'graph_labeled.jsonl'
SMOKE_FOLDS_JSON = SMOKE_GRAPH_DIR / 'folds.json'
run_module_main(graph_assembly, ['build', '--judge-manifest', str(SMOKE_JUDGE_MANIFEST),
                                  '--shard', 'Shard1Smoke', '--output', str(SMOKE_RAW_GRAPH)])
run_module_main(graph_assembly, ['join-labels', '--graph', str(SMOKE_RAW_GRAPH),
                                  '--labels', str(GROUND_TRUTH_CSV), '--output', str(SMOKE_LABELED_GRAPH)])
run_module_main(graph_assembly, ['folds', '--graph', str(SMOKE_LABELED_GRAPH),
                                  '--k', '2', '--output', str(SMOKE_FOLDS_JSON)])

smoke_rows = graph_assembly.read_jsonl(SMOKE_LABELED_GRAPH)
smoke_pos_edges = [(r['source_id'], r['target_id']) for r in smoke_rows
                    if r.get('type') == 'edge' and r.get('label') == 'Positive']
print(f"\nSmoke graph: {sum(1 for r in smoke_rows if r.get('type') == 'node')} nodes, "
      f"{sum(1 for r in smoke_rows if r.get('type') == 'edge')} edges, "
      f"{len(smoke_pos_edges)} labeled Positive")
if not smoke_pos_edges:
    print("WARNING: no Positive-labeled edge survived to the graph -- label wiring couldn't "
          "be exercised this run (the chosen pair may not have survived the geometric filter "
          "even at breakpoint=0, e.g. a VGGT/alignment error). Worth a look, but not "
          "necessarily a plumbing bug -- check the stage-by-stage row counts above first.")

SMOKE_CHECKPOINT_DIR = SMOKE_WORK / 'checkpoint'
run_module_main(train_negnet, [
    '--graph',    str(SMOKE_LABELED_GRAPH),
    '--folds',    str(SMOKE_FOLDS_JSON),
    '--val-fold', '0',
    '--features', str(SMOKE_FEATURE_CACHE),
    '--epochs',   '20',
    '--mp-rounds', '3',
    '--save-dir', str(SMOKE_CHECKPOINT_DIR),
])

smoke_ckpt = SMOKE_CHECKPOINT_DIR / 'negnet_best.pt'
assert smoke_ckpt.exists(), f"Smoke training did not produce a checkpoint at {smoke_ckpt}"
print(f"\nSMOKE TEST PASSED: full pipeline ran end-to-end, checkpoint written to {smoke_ckpt}")


## Full Shard 1 run

Each stage below checks whether its output already exists on Drive (staged from the earlier
raw-DINOv3 ablation run) and reuses it if so; it only falls back to actually running the
stage fresh if that Drive data is missing. Either way, the cell prints clearly which branch
it took.


In [ ]:
# Step 1 -- Stage 1 retrieval (raw DINOv3), reuse-or-run
if DINOV3RAW_RETRIEVAL_MANIFEST.exists() and DINOV3RAW_FEATURE_CACHE.exists():
    print(f"REUSING existing Stage 1 retrieval:\n  {DINOV3RAW_RETRIEVAL_MANIFEST}\n  {DINOV3RAW_FEATURE_CACHE}")
    STAGE1_MANIFEST = DINOV3RAW_RETRIEVAL_MANIFEST
    STAGE1_FEATURE_CACHE = DINOV3RAW_FEATURE_CACHE
else:
    print("Stage 1 retrieval not found on Drive -- running retrieve.py fresh (full corpus, "
          "expect several minutes on a T4).")
    STAGE1_OUT = LOCAL_WORK / 'stage1_retrieval'
    import retrieve
    importlib.reload(retrieve)
    retrieve.main([
        '--model-definition',   str(LOCAL_WORK / '_local' / 'ModelComboDINOv3Raw.py'),
        '--weights',            'none',
        '--source',             str(LOCAL_SOURCE_DIR),
        '--target',             str(LOCAL_TARGET_DIR),
        '--output-dir',         str(STAGE1_OUT),
        '--manifest-name',      'dinov3raw_retrieval_manifest.jsonl',
        '--feature-cache-name', 'dinov3raw_feature_cache.npz',
        '--topx',               '10',
        '--device',             'auto',
    ])
    STAGE1_MANIFEST = STAGE1_OUT / 'dinov3raw_retrieval_manifest.jsonl'
    STAGE1_FEATURE_CACHE = STAGE1_OUT / 'dinov3raw_feature_cache.npz'
    RETRIEVAL_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(STAGE1_MANIFEST), str(DINOV3RAW_RETRIEVAL_MANIFEST))
    shutil.copy(str(STAGE1_FEATURE_CACHE), str(DINOV3RAW_FEATURE_CACHE))
    print(f"Saved fresh Stage 1 outputs to Drive: {RETRIEVAL_OUT}")


In [ ]:
# Step 2 & 3 -- Stage 2 geometry filter + Stage 3 VGGT/pose scoring (raw DINOv3,
# Shard 1), reuse-or-run
if DINOV3RAW_S1_POSE_MANIFEST.exists():
    print(f"REUSING existing Stage 2+3 output:\n  {DINOV3RAW_S1_POSE_MANIFEST}\n  {DINOV3RAW_S1_VGGT_CANDIDATES}")
    if not DINOV3RAW_S1_VGGT_CANDIDATES.exists():
        raise FileNotFoundError(
            f"Missing on Drive: {DINOV3RAW_S1_VGGT_CANDIDATES}\n"
            f"{DINOV3RAW_S1_POSE_MANIFEST} alone is only the Stage-3 fresh-compute residual "
            "(98 rows, ~3 Positive) -- combining it with vggt_candidates_manifest.jsonl (731 "
            "rows, full ASpan-2D evidence + inherited judgements) is required to get the "
            "complete ~326-Positive picture instead of silently reproducing the degenerate-"
            "fold bug this fix exists for. Upload it and re-run."
        )

    # Combine: vggt_candidates_manifest.jsonl (731 rows, every Filter-1-survivor, ASpan-2D
    # evidence on all of them) is the base; pose_scored_manifest.jsonl (98 rows, the pairs
    # that needed FRESH Stage-3 compute) enriches whichever of those 731 it covers with full
    # VGGT/pose evidence too. The other ~633 candidates inherited an already-known judgement
    # from baseline's own review and never got a fresh Stage-3 row -- they still carry real
    # ASpan-2D evidence and real ground_truth.csv labels, just no VGGT/pose fields.
    # graph_assembly.py's EVIDENCE_KEYS extraction and negnet.py's per-field missing-value
    # handling already cope with that gracefully -- no code change needed downstream.
    pose_scored_by_pair = {}
    with open(str(DINOV3RAW_S1_POSE_MANIFEST), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                row = json.loads(line)
                pose_scored_by_pair[(row['source_id'], row['target_id'])] = row

    combined_rows = []
    n_enriched = 0
    with open(str(DINOV3RAW_S1_VGGT_CANDIDATES), encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            key = (row['source_id'], row['target_id'])
            if key in pose_scored_by_pair:
                combined_rows.append(pose_scored_by_pair[key])
                n_enriched += 1
            else:
                combined_rows.append(row)

    COMBINED_JUDGE_MANIFEST = LOCAL_WORK / 'dinov3raw_shard1_combined_judge_manifest.jsonl'
    COMBINED_JUDGE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    with open(str(COMBINED_JUDGE_MANIFEST), 'w', encoding='utf-8') as f:
        for row in combined_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f"Combined judge manifest: {len(combined_rows)} candidates "
          f"({n_enriched} with fresh Stage-3 evidence, {len(combined_rows) - n_enriched} "
          f"ASpan-2D-only with an inherited judgement) -> {COMBINED_JUDGE_MANIFEST}")
    STAGE3_JUDGE_MANIFEST = COMBINED_JUDGE_MANIFEST
else:
    print("Stage 2/3 output not found on Drive -- running geometry_filter.py + "
          "vggt_signals.py + pose_scoring.py fresh for Shard 1.")

    if not SOURCE_SHARD_ASSIGNMENT_JSON.exists():
        raise FileNotFoundError(
            f"Missing on Drive: {SOURCE_SHARD_ASSIGNMENT_JSON}\n"
            "Needed only for this fallback branch (Stage 2/3 reuse data was missing) -- "
            "upload NCR/shard_membership/source_shard_assignment.json and re-run."
        )
    source_shard_assignment = json.loads(SOURCE_SHARD_ASSIGNMENT_JSON.read_text(encoding='utf-8'))['assignment']
    shard1_true_sources = {sid for sid, m in source_shard_assignment.items() if m['shard'] == 'Shard1'}
    print(f"Shard 1 true source count: {len(shard1_true_sources)}")

    def _rewrite_path(p, local_root):
        path = Path(p)
        if path.is_absolute():
            return p
        return str(local_root / path.name)

    def split_manifest_by_shard(manifest_path, shard_sources, out_path):
        rows = []
        with open(str(manifest_path), encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                if str(row.get('source_id', '')) not in shard_sources:
                    continue
                if 'source_path' in row:
                    row['source_path'] = _rewrite_path(row['source_path'], LOCAL_SOURCE_DIR.resolve())
                if 'target_path' in row:
                    row['target_path'] = _rewrite_path(row['target_path'], LOCAL_TARGET_DIR.resolve())
                rows.append(json.dumps(row, ensure_ascii=False))
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text('\n'.join(rows) + ('\n' if rows else ''), encoding='utf-8')
        return len(rows)

    STAGE2_INPUT = LOCAL_WORK / 'stage2_shard1_candidates.jsonl'
    n_s1 = split_manifest_by_shard(STAGE1_MANIFEST, shard1_true_sources, STAGE2_INPUT)
    print(f"Shard 1 candidate rows: {n_s1}")

    STAGE2_OUT = LOCAL_WORK / 'stage2_geometry_shard1'
    import geometry_filter
    importlib.reload(geometry_filter)
    geometry_filter.main([
        '--input-manifest',   str(STAGE2_INPUT),
        '--output-dir',       str(STAGE2_OUT),
        '--breakpoint-value', '50',
        '--aspanpath',        ASPANPATH,
        '--weights_path',     str(WEIGHTS_PATH),
        '--config_path',      CONFIG_PATH,
        '--device',           'auto',
        '--progress-every',   '200',
    ])

    STAGE3_VGGT_OUT = LOCAL_WORK / 'stage3_vggt_shard1'
    import vggt_signals
    importlib.reload(vggt_signals)
    vggt_signals.main([
        '--input-manifest', str(STAGE2_OUT / 'vggt_candidates_manifest.jsonl'),
        '--output-dir',     str(STAGE3_VGGT_OUT),
        '--checkpoint',     str(VGGT_CHECKPOINT),
        '--device',         'auto',
        '--progress-every', '10',
    ])

    STAGE3_POSE_OUT = LOCAL_WORK / 'stage3_pose_shard1'
    import pose_scoring
    importlib.reload(pose_scoring)
    pose_scoring.main([
        '--input-manifest', str(STAGE3_VGGT_OUT / 'vggt_judged_manifest.jsonl'),
        '--output-dir',     str(STAGE3_POSE_OUT),
    ])

    STAGE3_JUDGE_MANIFEST = STAGE3_POSE_OUT / 'pose_scored_manifest.jsonl'
    DINOV3RAW_S1_POSE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(STAGE3_JUDGE_MANIFEST), str(DINOV3RAW_S1_POSE_MANIFEST))
    print(f"Saved fresh Stage 2+3 output to Drive: {DINOV3RAW_S1_POSE_MANIFEST}")


In [ ]:
# Step 4 -- Graph assembly: build, join-labels (against the full, consolidated ground
# truth, not just the original match_manifest_shard1.csv), folds
import graph_assembly
importlib.reload(graph_assembly)

GRAPH_DIR = LOCAL_WORK / 'graph_shard1'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
RAW_GRAPH = GRAPH_DIR / 'shard1_graph.jsonl'
LABELED_GRAPH = GRAPH_DIR / 'shard1_graph_labeled.jsonl'
FOLDS_JSON = GRAPH_DIR / 'shard1_folds.json'

run_module_main(graph_assembly, ['build', '--judge-manifest', str(STAGE3_JUDGE_MANIFEST),
                                  '--shard', 'Shard1', '--output', str(RAW_GRAPH)])
run_module_main(graph_assembly, ['join-labels', '--graph', str(RAW_GRAPH),
                                  '--labels', str(GROUND_TRUTH_CSV), '--output', str(LABELED_GRAPH)])
run_module_main(graph_assembly, ['folds', '--graph', str(LABELED_GRAPH),
                                  '--k', '5', '--output', str(FOLDS_JSON)])

graph_rows = graph_assembly.read_jsonl(LABELED_GRAPH)
n_nodes = sum(1 for r in graph_rows if r.get('type') == 'node')
n_edges = sum(1 for r in graph_rows if r.get('type') == 'edge')
n_labeled = sum(1 for r in graph_rows if r.get('type') == 'edge' and r.get('label'))
n_positive = sum(1 for r in graph_rows if r.get('type') == 'edge' and r.get('label') == 'Positive')
print(f"\nShard 1 graph: {n_nodes} nodes, {n_edges} edges "
      f"({n_labeled} labeled, {n_positive} Positive)")


In [ ]:
# Step 4 (cont.) -- record NEG-Net's training-exposure registry. Leakage prevention for
# NEG-Net (a transductive, node-embedding model) is a cross-RUN concept, not something this
# graph needs to enforce on itself: the model forward-passes over every node in one batch
# during training, so "has this exact node been seen before" only matters once a SEPARATE
# graph (a future Shard 2/3+ evaluation) gets built and might reuse it. No such run exists yet
# in this pass (in-shard folds only), so nothing here blocks or drops anything. Instead, this
# cell permanently RECORDS every node (and edge, for the audit trail) this training graph
# touches, so a future evaluation run can check itself against it and drop conflicts from ITS
# OWN report -- see NCR/negnet_training_exposure/README.md for the full mechanism and
# check_negnet_exposure.py for the tool that consumes this file.
#
# No shard identity of any kind is recorded here -- leakage prevention is keyed purely on
# node_id exposure, entirely independent of which shard a node belongs to (see the README for
# why: an earlier family-reassignment approach tried to prevent leakage by enforcing shard
# disjointness in advance; this record-and-drop-later approach replaced it and needs no shard
# concept at all).
graph_node_ids = [r['node_id'] for r in graph_rows if r.get('type') == 'node']
node_side = {r['node_id']: r.get('side') for r in graph_rows if r.get('type') == 'node'}

node_registry = {nid: {'side': node_side.get(nid)} for nid in graph_node_ids}

edge_registry = [
    {
        'source_id': r['source_id'],
        'target_id': r['target_id'],
        'origin': r.get('origin'),
        'label': r.get('label'),
        'missing_evidence': bool(r.get('missing_evidence')),
    }
    for r in graph_rows if r.get('type') == 'edge'
]

EXPOSURE_REGISTRY = {
    'purpose': (
        "Permanent record of every node/edge NEG-Net's Shard1 raw-DINOv3 training graph "
        "touched. A future evaluation graph (Shard 2, 3, ...) must check every one of its own "
        "nodes against this file's 'nodes' keys and DROP (delete the node entirely, not just "
        "its edges -- see check_negnet_exposure.py) any hit before evaluating, to avoid "
        "train/val leakage through this transductive model's message passing."
    ),
    'generated': time.strftime('%Y-%m-%d'),
    'backbone': 'raw-DINOv3 (ModelComboDINOv3Raw)',
    'shard': 'Shard1',
    'source_graph': str(LABELED_GRAPH),
    # Must match Step 7's CONFIGS tags below -- duplicated here since this cell runs first.
    'checkpoints_trained_on_this_exposure_set': ['mp0', 'mp3', 'mp3_noloss'],
    'node_count': len(node_registry),
    'edge_count': len(edge_registry),
    'nodes': node_registry,
    'edges': edge_registry,
}

print(f"Graph nodes: {len(graph_node_ids)}")

EXPOSURE_REGISTRY_PATH = GRAPH_DIR / 'shard1_training_exposure.json'
EXPOSURE_REGISTRY_PATH.write_text(
    json.dumps(EXPOSURE_REGISTRY, indent=2, ensure_ascii=False), encoding='utf-8')
sync_dir_to_drive(GRAPH_DIR, NEGNET_OUT / 'training_exposure', ['shard1_training_exposure.json'])
print(f"\nExposure registry written: {EXPOSURE_REGISTRY_PATH} "
      f"({len(node_registry)} nodes, {len(edge_registry)} edges) -- synced to Drive.")
print("Pull this into NCR/negnet_training_exposure/ after this run completes, so future "
      "sessions/notebooks can see it without re-running Colab.")


In [ ]:
# Step 5/6 -- Node features = Stage 1's feature cache directly (no separate dump
# script -- ModelComboDINOv3Raw has no head, so retrieval-time output already is the node
# feature). train_negnet.py infers NODE_DIM dynamically now (verified present earlier) --
# sanity-print it here before spending GPU time on training.
import train_negnet
importlib.reload(train_negnet)

_lookup_preview = train_negnet.load_feature_lookup([str(STAGE1_FEATURE_CACHE)])
_node_dim_preview = train_negnet.infer_node_dim(_lookup_preview)
print(f"Feature cache: {STAGE1_FEATURE_CACHE}")
print(f"  {len(_lookup_preview)} cached vectors, inferred NODE_DIM = {_node_dim_preview}")
assert _node_dim_preview == 1024, (
    f"Expected 1024-d raw DINOv3 features, got {_node_dim_preview} -- check STAGE1_FEATURE_CACHE."
)


In [ ]:
# Step 6.5 -- Real-data smoke test: one short, DISCARDED training run on the REAL Shard 1
# graph/features, before committing to the full 3-config training below. Distinct from Step
# 0.5 (which validates the whole pipeline on tiny FAKE images, early, before Stage 1 even
# runs for real): this one exercises the actual graph's real scale, class balance, and
# missingness pattern -- things a 5-pair fake-image test can't reach (e.g. degenerate fold
# splits, NaN losses on the real feature distribution). Given the real Shard 1 graph is small
# (~98 candidate edges per the existing rawdino ablation), this is cheap regardless -- a
# ~2M-param model, full-batch.
REAL_SMOKE_DIR = LOCAL_WORK / 'real_data_smoke_test'

run_module_main(train_negnet, [
    '--graph',     str(LABELED_GRAPH),
    '--folds',     str(FOLDS_JSON),
    '--val-fold',  '0',
    '--features',  str(STAGE1_FEATURE_CACHE),
    '--epochs',    '30',
    '--mp-rounds', '3',
    '--save-dir',  str(REAL_SMOKE_DIR),
])

real_smoke_ckpt = REAL_SMOKE_DIR / 'negnet_best.pt'
real_smoke_config = json.loads((REAL_SMOKE_DIR / 'run_config.json').read_text(encoding='utf-8'))
assert real_smoke_ckpt.exists(), f"Real-data smoke test did not produce a checkpoint at {real_smoke_ckpt}"
best_f1 = real_smoke_config['best_val']['f1']
assert best_f1 == best_f1, "Real-data smoke test produced a NaN val F1 -- do not proceed to Step 7."

print(f"Real-data smoke test: best val F1 {best_f1:.3f} @ epoch {real_smoke_config['best_val']['epoch']}")
print(f"REAL-DATA SMOKE TEST PASSED -- checkpoint at {real_smoke_ckpt} (discarded, not synced to Drive).")
print("Safe to proceed to the full 3-config training run below.")


In [ ]:
# Step 7 -- Train NEG-Net Tier-0: all 3 attribution-ladder configs (per
# train_negnet.py's own docstring), each frozen and saved to its own labeled directory --
# not a single winner-take-all model. This run is intended to be the only one: per project
# decision these weights are not retrained afterward.
CHECKPOINTS_OUT = LOCAL_WORK / 'checkpoints'
CONFIGS = [
    ('mp0',        ['--mp-rounds', '0']),                                    # pair-MLP baseline
    ('mp3',        ['--mp-rounds', '3']),                                    # + message passing (default)
    ('mp3_noloss', ['--mp-rounds', '3', '--lambda-tri', '0', '--lambda-nest', '0']),  # consistency losses ablated
]

import train_negnet
importlib.reload(train_negnet)

for tag, extra_args in CONFIGS:
    save_dir = CHECKPOINTS_OUT / tag
    print(f"\n{'=' * 70}\nTraining config: {tag}  ({' '.join(extra_args)})\n{'=' * 70}")
    run_module_main(train_negnet, [
        '--graph',    str(LABELED_GRAPH),
        '--folds',    str(FOLDS_JSON),
        '--val-fold', '0',
        '--features', str(STAGE1_FEATURE_CACHE),
        '--save-dir', str(save_dir),
        *extra_args,
    ])
    run_config = json.loads((save_dir / 'run_config.json').read_text(encoding='utf-8'))
    print(f"{tag}: best val F1 {run_config['best_val']['f1']:.3f} @ epoch {run_config['best_val']['epoch']}")

# Sync every config's checkpoint to Drive as a permanent, frozen artifact.
for tag, _ in CONFIGS:
    src = CHECKPOINTS_OUT / tag
    dst = NEGNET_CHECKPOINTS_DRIVE / tag
    sync_dir_to_drive(src, dst, ['negnet_best.pt', 'standardizer.json', 'run_config.json'])


In [ ]:
# Final report -- all 3 attribution-ladder configs, side by side
print(f"{'config':<12} {'mp_rounds':<10} {'lambda_tri':<11} {'lambda_nest':<12} {'best_f1':<8} best_epoch")
for tag, _ in CONFIGS:
    run_config = json.loads((CHECKPOINTS_OUT / tag / 'run_config.json').read_text(encoding='utf-8'))
    print(f"{tag:<12} {run_config['mp_rounds']:<10} {run_config['lambda_tri']:<11} "
          f"{run_config['lambda_nest']:<12} {run_config['best_val']['f1']:<8.3f} "
          f"{run_config['best_val']['epoch']}")

print(f"\nAll checkpoints frozen and saved to: {NEGNET_CHECKPOINTS_DRIVE}")
print("Per project decision, these weights are NOT meant to be retrained -- this run is final.")
